<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/15-latent-variable-flow-energy-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Latent-Variable, Flow, and Energy-Based Models** {#latent-variable-flow-energy-models}

The autoregressive construction in Chapter 14 makes likelihood tractable by choosing an order. This chapter studies three different answers to the same density-modeling problem. **Latent-variable models** explain observations through hidden variables; **normalizing flows** transform a simple density through an invertible map; **energy-based models** assign low scalar energy to plausible configurations without requiring an immediately tractable normalizer.

![Latent-variable, flow, and energy-based models make different density and inference commitments.](assets/dl15-model-family-map.svg){fig-align="center" width="78%" fig-alt="Three panels compare latent-variable models, normalizing flows, and energy-based models by their representation of a data distribution."}

*Original synthesis based on [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), [RealNVP](https://arxiv.org/abs/1605.08803), and the [Energy-Based Learning tutorial](https://yann.lecun.org/exdb/publis/pdf/lecun-06.pdf).*

All experiments use scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), CC BY 4.0. The fixed 70/15/15 split is created once. Autoencoders and VAEs operate on the same 64-dimensional images; later flow, energy, and score models operate on a standardized two-dimensional autoencoder representation so their density geometry can be inspected directly. This low-dimensional stage is a mechanism demonstration, not a modern image-generation benchmark.

<details>
<summary><strong>PyTorch: establish the shared image and latent-density experiment</strong></summary>

```python
import copy
import math
import os
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")


def seed_everything(seed=1515):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).flatten(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1515, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1515,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]


def make_loader(images, batch_size=160, seed=1515):
    return DataLoader(
        TensorDataset(images), batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )


assert all_images.shape == (1797, 64)
assert len(set(train_idx) & set(test_idx)) == 0
assert train_x.min() >= 0 and train_x.max() <= 1
print({"split": (len(train_x), len(val_x), len(test_x)),
       "input shape": tuple(train_x.shape), "classes": int(all_labels.unique().numel())})
```

</details>

No labels enter representation-model training; labels are retained only for later diagnostic probes. Every learned normalization statistic is fit on the training representation and then reused on validation, test, and generated points.


### **Latent Variables and Hidden Structure** {#latent-variables-hidden-structure}

A latent-variable model introduces an unobserved variable $z$ so that

$$
p_{\theta}(x)=\int p_{\theta}(x\mid z)p(z)\,dz.
$$

$z$ can represent class, pose, style, topic, speaker identity, physical state, or another hidden cause. The factorization does not guarantee that individual coordinates acquire human-interpretable meanings: many latent representations produce the same marginal $p(x)$, and rotations or permutations can leave likelihood unchanged. Interpretability therefore requires assumptions, supervision, interventions, architectural bias, or evaluation against known factors.

Three questions determine whether a latent model is useful. **Representation:** which information about $x$ is retained in $z$? **Inference:** can $p(z\mid x)$ be computed or approximated? **Generation:** is there a specified prior from which a valid $z$ can be sampled? A deterministic embedding may answer the first question while leaving the other two undefined.

For the digit experiment, a two-dimensional code deliberately forces compression. It can preserve broad shape and class neighborhoods, but it cannot encode every stroke detail. Later density models will estimate the distribution of these codes; they model the autoencoder representation, not raw images directly.


### **Deterministic Autoencoders** {#deterministic-autoencoders}

An autoencoder learns an encoder $z=f_{\phi}(x)$ and decoder $\hat{x}=g_{\theta}(z)$ by minimizing reconstruction error. For fixed-variance Gaussian observations, mean squared error is proportional to negative log-likelihood:

$$
\mathcal{L}_{\mathrm{AE}}=\frac{1}{N}\sum_n\lVert x^{(n)}-g_{\theta}(f_{\phi}(x^{(n)}))\rVert_2^2.
$$

![A deterministic autoencoder compresses and reconstructs, but does not define a prior over its codes.](assets/dl15-autoencoder-bottleneck.svg){fig-align="center" width="76%" fig-alt="An input passes through an encoder, a two-dimensional bottleneck, and a decoder; a final warning notes that sampling has no defined latent prior."}

*Original teaching diagram.*

The bottleneck prevents a trivial identity map only when capacity and regularization are genuinely restrictive. An overcomplete network can memorize. Reconstruction quality also does not imply a smooth or sampleable latent space: arbitrary points between encoded examples may decode poorly because training never constrained those regions.

<details>
<summary><strong>PyTorch: train a two-dimensional deterministic autoencoder</strong></summary>

```python
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.ReLU(), nn.Linear(48, 64), nn.Sigmoid())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


def reconstruction_mse(model, images):
    model.eval()
    with torch.no_grad():
        reconstruction, _ = model(images)
    return float(F.mse_loss(reconstruction, images))


seed_everything(1520)
autoencoder = Autoencoder(latent_dim=2)
optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=3e-3, weight_decay=1e-5)
loader = make_loader(train_x, seed=1520)
best_state, best_val_mse = copy.deepcopy(autoencoder.state_dict()), float("inf")
for _ in range(90):
    autoencoder.train()
    for (batch,) in loader:
        optimizer.zero_grad()
        reconstruction, _ = autoencoder(batch)
        loss = F.mse_loss(reconstruction, batch)
        loss.backward()
        optimizer.step()
    val_mse = reconstruction_mse(autoencoder, val_x)
    if val_mse < best_val_mse:
        best_val_mse, best_state = val_mse, copy.deepcopy(autoencoder.state_dict())
autoencoder.load_state_dict(best_state)

autoencoder.eval()
with torch.no_grad():
    train_z_raw = autoencoder.encoder(train_x)
    val_z_raw = autoencoder.encoder(val_x)
    test_z_raw = autoencoder.encoder(test_x)
latent_mean = train_z_raw.mean(0)
latent_std = train_z_raw.std(0).clamp_min(1e-5)
train_z = (train_z_raw - latent_mean) / latent_std
val_z = (val_z_raw - latent_mean) / latent_std
test_z = (test_z_raw - latent_mean) / latent_std

test_ae_mse = reconstruction_mse(autoencoder, test_x)
assert test_ae_mse < 0.08
assert torch.allclose(train_z.mean(0), torch.zeros(2), atol=1e-5)
print({"best validation MSE": round(best_val_mse, 4), "test MSE": round(test_ae_mse, 4),
       "standardized latent mean": train_z.mean(0).round(decimals=3).tolist(),
       "standardized latent std": train_z.std(0).round(decimals=3).tolist()})
```

</details>

The standardization is fit only on training codes. The resulting $[B,2]$ tensors support interpretable density experiments, but the two-dimensional bottleneck sacrifices reconstruction detail. A larger latent code would improve reconstruction and make later geometry harder to visualize.


### **Denoising and Sparse Autoencoders** {#denoising-sparse-autoencoders}

A denoising autoencoder receives a corrupted $\tilde{x}\sim q(\tilde{x}\mid x)$ and predicts the clean $x$. It cannot solve the task by copying each input coordinate, so it must learn regularities that distinguish signal from corruption. [Denoising autoencoders](https://www.jmlr.org/papers/v11/vincent10a.html) connect local denoising behavior to useful representations.

A sparse autoencoder penalizes latent activity, for example with $\lambda\lVert z\rVert_1$, or matches average activation to a small target. Sparsity can produce selective features and limit effective capacity, but too much regularization discards information and creates dead units.

![Denoising and sparsity constrain an autoencoder to preserve stable structure rather than copy its input.](assets/dl15-denoising-sparse.svg){fig-align="center" width="76%" fig-alt="A corrupted input is encoded into a sparse representation, decoded toward the clean target, and evaluated for robustness and sparsity."}

*Original process diagram based on the denoising objective described by [Vincent et al.](https://www.jmlr.org/papers/v11/vincent10a.html).*

<details>
<summary><strong>PyTorch: train one sparse denoising autoencoder and test corruption robustness</strong></summary>

```python
class SparseDenoisingAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, latent_dim), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.ReLU(), nn.Linear(48, 64), nn.Sigmoid())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


def corrupt(images, generator, mask_probability=0.18, noise_std=0.12):
    mask = torch.rand(images.shape, generator=generator) > mask_probability
    noise = noise_std * torch.randn(images.shape, generator=generator)
    return (images * mask + noise).clamp(0.0, 1.0)


seed_everything(1530)
denoising_ae = SparseDenoisingAE()
optimizer = torch.optim.AdamW(denoising_ae.parameters(), lr=2e-3, weight_decay=1e-5)
loader = make_loader(train_x, seed=1530)
noise_generator = torch.Generator().manual_seed(1530)
for _ in range(70):
    denoising_ae.train()
    for (clean_batch,) in loader:
        noisy_batch = corrupt(clean_batch, noise_generator)
        optimizer.zero_grad()
        reconstruction, latent = denoising_ae(noisy_batch)
        loss = F.mse_loss(reconstruction, clean_batch) + 8e-4 * latent.abs().mean()
        loss.backward()
        optimizer.step()

test_noise = corrupt(test_x, torch.Generator().manual_seed(1531))
denoising_ae.eval()
with torch.no_grad():
    denoised, test_sparse_z = denoising_ae(test_noise)
    noisy_mse = float(F.mse_loss(test_noise, test_x))
    denoised_mse = float(F.mse_loss(denoised, test_x))
    inactive_fraction = float((test_sparse_z < 1e-3).float().mean())

assert denoised_mse < noisy_mse
print({"corrupted-input MSE": round(noisy_mse, 4), "denoised MSE": round(denoised_mse, 4),
       "near-zero latent fraction": round(inactive_fraction, 3)})
```

</details>

The corruption distribution defines the invariance being taught. Masking and Gaussian noise are reasonable for this demonstration, but they do not represent every real sensor failure. Denoising performance must be tested under corruptions relevant to deployment; otherwise robustness claims are misplaced.


### **Variational Autoencoders** {#variational-autoencoders}

A variational autoencoder (VAE) turns an autoencoder into a probabilistic latent-variable model. The generative model specifies $p(z)$ and $p_{\theta}(x\mid z)$. Because the posterior

$$
p_{\theta}(z\mid x)=\frac{p_{\theta}(x\mid z)p(z)}{p_{\theta}(x)}
$$

is generally intractable, an encoder $q_{\phi}(z\mid x)$ approximates it. A common choice is a diagonal Gaussian with encoder-produced $\mu_{\phi}(x)$ and $\log\sigma_{\phi}^2(x)$. Generation samples $z\sim p(z)=\mathcal{N}(0,I)$ and then samples or decodes from $p_{\theta}(x\mid z)$.

![A VAE joins an approximate inference model to a probabilistic decoder and prior.](assets/dl15-vae-graph.svg){fig-align="center" width="76%" fig-alt="An observed input enters approximate inference q, producing latent z regularized by a prior; the decoder maps z to a generated observation."}

*Original graphical explanation based on [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114).*

The observation model is a modeling decision. Here the decoder outputs the mean of a fixed-variance Gaussian over normalized pixel intensities. This is computationally transparent but can blur multimodal details. A Bernoulli likelihood is appropriate for binary pixels, not automatically for arbitrary continuous images.

<details>
<summary><strong>PyTorch: train a Gaussian-decoder variational autoencoder</strong></summary>

```python
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.latent_dim = latent_dim
        self.backbone = nn.Sequential(nn.Linear(64, 56), nn.ReLU())
        self.mu = nn.Linear(56, latent_dim)
        self.logvar = nn.Linear(56, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 56), nn.ReLU(), nn.Linear(56, 64), nn.Sigmoid())

    def encode(self, images):
        hidden = self.backbone(images)
        return self.mu(hidden), self.logvar(hidden).clamp(-8.0, 6.0)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def forward(self, images):
        mu, logvar = self.encode(images)
        latent = self.reparameterize(mu, logvar)
        return self.decoder(latent), mu, logvar


decoder_sigma = 0.20


def vae_terms(reconstruction, target, mu, logvar):
    reconstruction_nll = (
        0.5 * ((target - reconstruction) / decoder_sigma).pow(2)
        + math.log(decoder_sigma) + 0.5 * math.log(2 * math.pi)
    ).sum(dim=1)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
    return reconstruction_nll, kl


def train_vae(beta, seed, epochs=65, warmup=False):
    seed_everything(seed)
    model = VariationalAutoencoder()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
    loader = make_loader(train_x, seed=seed)
    for epoch in range(epochs):
        model.train()
        effective_beta = beta * min(1.0, (epoch + 1) / 18) if warmup else beta
        for (batch,) in loader:
            optimizer.zero_grad()
            reconstruction, mu, logvar = model(batch)
            reconstruction_nll, kl = vae_terms(reconstruction, batch, mu, logvar)
            loss = (reconstruction_nll + effective_beta * kl).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    return model


def evaluate_vae(model, images):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(images)
        reconstruction = model.decoder(mu)
        reconstruction_nll, kl = vae_terms(reconstruction, images, mu, logvar)
    return float(reconstruction_nll.mean()), float(kl.mean()), mu, logvar


vae = train_vae(beta=1.0, seed=1540, warmup=True)
vae_reconstruction_nll, vae_kl, test_mu, test_logvar = evaluate_vae(vae, test_x)
with torch.no_grad():
    prior_samples = vae.decoder(torch.randn(40, vae.latent_dim))

assert prior_samples.shape == (40, 64)
assert math.isfinite(vae_reconstruction_nll + vae_kl)
print({"test reconstruction NLL": round(vae_reconstruction_nll, 2),
       "test KL": round(vae_kl, 2), "prior sample range":
       (round(float(prior_samples.min()), 3), round(float(prior_samples.max()), 3))})
```

</details>

A finite objective and valid range do not prove sample quality. The fixed decoder variance, latent dimension, prior, and network capacity all shape the learned solution. Report these choices with the ELBO rather than presenting “VAE loss” as a model-independent metric.

The reconstruction NLL can be negative here because a continuous probability **density** may exceed one when its variance is small; only its integral must equal one. This is not a probability below zero. Changing `decoder_sigma` changes the numerical density and therefore changes the reported NLL.


### **Evidence Lower Bound** {#evidence-lower-bound}

Insert any approximate posterior $q_{\phi}(z\mid x)$ and apply Jensen's inequality:

$$
\log p_{\theta}(x)
\ge
\mathbb{E}_{q_{\phi}(z\mid x)}[\log p_{\theta}(x\mid z)]
-\mathrm{KL}(q_{\phi}(z\mid x)\|p(z))
=\mathcal{L}_{\mathrm{ELBO}}.
$$

The exact gap is

$$
\log p_{\theta}(x)-\mathcal{L}_{\mathrm{ELBO}}
=\mathrm{KL}(q_{\phi}(z\mid x)\|p_{\theta}(z\mid x))\ge0.
$$

Thus a poor ELBO can result from a poor generative model, an inaccurate inference family, or both. The reconstruction term encourages $z$ to preserve information about $x$; the prior KL limits information and makes encoded regions compatible with prior sampling.

![The ELBO combines expected reconstruction with a KL cost and lower-bounds log evidence.](assets/dl15-elbo.svg){fig-align="center" width="78%" fig-alt="Expected reconstruction minus posterior-to-prior KL equals the evidence lower bound."}

*Original ELBO decomposition based on [Kingma and Welling](https://arxiv.org/abs/1312.6114).*

<details>
<summary><strong>PyTorch: decompose the held-out ELBO per observation</strong></summary>

```python
vae.eval()
with torch.no_grad():
    mu, logvar = vae.encode(test_x)
    generator = torch.Generator().manual_seed(1550)
    epsilon = torch.randn(mu.shape, generator=generator)
    latent = mu + torch.exp(0.5 * logvar) * epsilon
    reconstruction = vae.decoder(latent)
    heldout_reconstruction_nll, heldout_kl = vae_terms(reconstruction, test_x, mu, logvar)
    negative_elbo = heldout_reconstruction_nll + heldout_kl

assert torch.all(heldout_kl >= -1e-5)
assert torch.allclose(negative_elbo, heldout_reconstruction_nll + heldout_kl)
print({"mean negative ELBO": round(float(negative_elbo.mean()), 2),
       "reconstruction contribution": round(float(heldout_reconstruction_nll.mean()), 2),
       "KL contribution": round(float(heldout_kl.mean()), 2),
       "KL share of absolute objective": round(float(heldout_kl.abs().mean() /
                                                       (heldout_reconstruction_nll.abs().mean() + heldout_kl.abs().mean())), 3)})
```

</details>

An ELBO is a lower bound under the stated likelihood and variational family. It is not directly comparable when image scaling, decoder variance, dequantization, or likelihood family differs. Multiple importance-weighted samples can tighten evaluation bounds, but a tighter estimator does not repair a poor model.


### **The Reparameterization Trick** {#reparameterization-trick}

Sampling $z\sim q_{\phi}(z\mid x)$ appears to block ordinary backpropagation. For a diagonal Gaussian, write

$$
\epsilon\sim\mathcal{N}(0,I),\qquad
z=\mu_{\phi}(x)+\sigma_{\phi}(x)\odot\epsilon.
$$

The randomness is now isolated in parameter-free $\epsilon$, while $z$ is differentiable with respect to $\mu$ and $\sigma$. This produces a pathwise gradient estimator, usually lower variance than score-function estimators for continuous reparameterizable distributions.

![Reparameterization expresses a stochastic latent as a deterministic function of parameters and parameter-free noise.](assets/dl15-reparameterization.svg){fig-align="center" width="76%" fig-alt="Encoder outputs mu and log variance; external Gaussian epsilon enters z equals mu plus sigma epsilon; gradients flow through z to the encoder."}

*Original computational-graph diagram based on [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114).*

<details>
<summary><strong>PyTorch: verify pathwise gradients and contrast sample with rsample</strong></summary>

```python
seed_everything(1560)
demo_mu = torch.tensor([[0.3, -0.2]], requires_grad=True)
demo_logvar = torch.tensor([[0.1, -0.4]], requires_grad=True)
distribution = torch.distributions.Normal(demo_mu, torch.exp(0.5 * demo_logvar))

pathwise_z = distribution.rsample()
pathwise_loss = pathwise_z.pow(2).sum()
pathwise_loss.backward()
mu_gradient = demo_mu.grad.detach().clone()
logvar_gradient = demo_logvar.grad.detach().clone()

demo_mu_2 = torch.tensor([[0.3, -0.2]], requires_grad=True)
ordinary_sample = torch.distributions.Normal(demo_mu_2, torch.ones_like(demo_mu_2)).sample()

assert pathwise_z.requires_grad
assert not ordinary_sample.requires_grad
assert mu_gradient.abs().sum() > 0 and logvar_gradient.abs().sum() > 0
print({"rsample requires grad": pathwise_z.requires_grad,
       "sample requires grad": ordinary_sample.requires_grad,
       "mu gradient": mu_gradient.round(decimals=3).tolist(),
       "log-variance gradient": logvar_gradient.round(decimals=3).tolist()})
```

</details>

Reparameterization does not remove Monte Carlo variance; it changes the estimator. Discrete variables generally need other tools such as relaxations, straight-through estimators, marginalization, or score-function gradients. Each introduces its own bias or variance trade-off.


### **Beta-VAE and Disentanglement** {#beta-vae-disentanglement}

$\beta$-VAE modifies the objective:

$$
\mathcal{L}_{\beta}=\mathbb{E}_{q(z\mid x)}[-\log p(x\mid z)]
+\beta\,\mathrm{KL}(q(z\mid x)\|p(z)).
$$

$\beta>1$ restricts latent capacity more strongly and can encourage factorized codes, but usually worsens reconstruction. [The original $\beta$-VAE study](https://openreview.net/references/pdf?id=B1-vyHvOe) evaluates disentanglement on data with known factors. Unsupervised disentanglement is not identifiable without inductive assumptions: visual axis traversals alone do not establish that coordinates correspond to true independent causes.

The experiment compares $\beta=0.25$, $1$, and $4$ on the same split. A linear label probe measures whether latent means retain digit-class information; it is a diagnostic, not part of VAE training and not a general disentanglement metric.

<details>
<summary><strong>PyTorch: measure the reconstruction, rate, and probe trade-off across beta</strong></summary>

```python
low_beta_vae = train_vae(beta=0.25, seed=1570, epochs=55)
high_beta_vae = train_vae(beta=4.0, seed=1571, epochs=55)
beta_models = {0.25: low_beta_vae, 1.0: vae, 4.0: high_beta_vae}
beta_results = {}
for beta, model in beta_models.items():
    reconstruction_nll, kl, _, _ = evaluate_vae(model, test_x)
    model.eval()
    with torch.no_grad():
        train_mu, _ = model.encode(train_x)
        test_mu_beta, _ = model.encode(test_x)
    probe = LogisticRegression(max_iter=500, random_state=1570).fit(train_mu.numpy(), train_y.numpy())
    probe_accuracy = accuracy_score(test_y.numpy(), probe.predict(test_mu_beta.numpy()))
    beta_results[beta] = {"reconstruction NLL": reconstruction_nll, "KL": kl,
                          "linear probe accuracy": probe_accuracy}

for beta, row in beta_results.items():
    print({"beta": beta, **{key: round(value, 3) for key, value in row.items()}})

assert beta_results[4.0]["KL"] < beta_results[0.25]["KL"]
```

</details>

The KL is often called the **rate**, while expected reconstruction cost is the **distortion**. Changing $\beta$ moves along a rate-distortion frontier rather than monotonically “improving” the representation. Choose the operating point for the downstream requirement and evaluate known factors, interventions, or task utility when claiming disentanglement.


### **Vector-Quantized VAE** {#vector-quantized-vae}

VQ-VAE maps an encoder output $z_e(x)$ to the nearest learned codebook vector $e_k$:

$$
k^{*}=\arg\min_k\lVert z_e(x)-e_k\rVert_2^2,qquad z_q(x)=e_{k^{*}}.
$$

The discrete index can be modeled by an autoregressive prior. The decoder sees the quantized vector. Because nearest-neighbor selection has zero or undefined gradients, the straight-through estimator copies decoder gradients to the encoder. Separate codebook and commitment losses move embeddings toward encoder outputs and keep encoder outputs near selected embeddings.

![VQ-VAE quantizes continuous encoder outputs with a learned discrete codebook before decoding.](assets/dl15-vqvae.svg){fig-align="center" width="76%" fig-alt="An encoder output selects its nearest codebook vector, producing a discrete code and straight-through input to the decoder."}

*Original mechanism diagram based on [Neural Discrete Representation Learning](https://arxiv.org/abs/1711.00937).*

<details>
<summary><strong>PyTorch: implement codebook lookup, straight-through gradients, and utilization</strong></summary>

```python
class VQVAE(nn.Module):
    def __init__(self, codebook_size=24, embedding_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, embedding_dim))
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        nn.init.uniform_(self.codebook.weight, -0.35, 0.35)
        self.decoder = nn.Sequential(nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.Sigmoid())

    def forward(self, images):
        encoded = self.encoder(images)
        distances = (encoded.pow(2).sum(1, keepdim=True)
                     - 2 * encoded @ self.codebook.weight.T
                     + self.codebook.weight.pow(2).sum(1).unsqueeze(0))
        indices = distances.argmin(dim=1)
        quantized = self.codebook(indices)
        straight_through = encoded + (quantized - encoded).detach()
        reconstruction = self.decoder(straight_through)
        return reconstruction, encoded, quantized, indices


seed_everything(1580)
vqvae = VQVAE()
loader = make_loader(train_x, seed=1580)
# First learn a continuous bottleneck so that codebook initialization sees the data geometry.
pretrain_optimizer = torch.optim.AdamW(
    list(vqvae.encoder.parameters()) + list(vqvae.decoder.parameters()), lr=2e-3
)
for _ in range(35):
    for (batch,) in loader:
        pretrain_optimizer.zero_grad()
        encoded = vqvae.encoder(batch)
        reconstruction = vqvae.decoder(encoded)
        F.mse_loss(reconstruction, batch).backward()
        pretrain_optimizer.step()

with torch.no_grad():
    initial_codes = vqvae.encoder(train_x).numpy()
centers = KMeans(
    n_clusters=vqvae.codebook.num_embeddings, n_init=10, random_state=1580
).fit(initial_codes).cluster_centers_
with torch.no_grad():
    vqvae.codebook.weight.copy_(torch.tensor(centers, dtype=torch.float32))

optimizer = torch.optim.AdamW(vqvae.parameters(), lr=1e-3, weight_decay=1e-5)
for _ in range(65):
    vqvae.train()
    for (batch,) in loader:
        optimizer.zero_grad()
        reconstruction, encoded, quantized, _ = vqvae(batch)
        reconstruction_loss = F.mse_loss(reconstruction, batch)
        codebook_loss = F.mse_loss(quantized, encoded.detach())
        commitment_loss = F.mse_loss(encoded, quantized.detach())
        loss = reconstruction_loss + codebook_loss + 0.25 * commitment_loss
        loss.backward()
        optimizer.step()

vqvae.eval()
with torch.no_grad():
    vq_reconstruction, _, _, vq_indices = vqvae(test_x)
    vq_mse = float(F.mse_loss(vq_reconstruction, test_x))
    usage = torch.bincount(vq_indices, minlength=vqvae.codebook.num_embeddings).float()
    usage_probability = usage / usage.sum()
    codebook_perplexity = float(torch.exp(-(usage_probability[usage > 0] *
                                             usage_probability[usage > 0].log()).sum()))

assert vq_indices.dtype == torch.long
print({"test reconstruction MSE": round(vq_mse, 4),
       "used codes": int((usage > 0).sum()),
       "codebook size": vqvae.codebook.num_embeddings,
       "codebook perplexity": round(codebook_perplexity, 2)})
assert int((usage > 0).sum()) >= 4
```

</details>

Codebook collapse occurs when only a few entries are used. Usage histograms and perplexity should be monitored; exponential-moving-average updates, code resets, larger batches, or commitment tuning can help. A VQ-VAE decoder alone is not a complete generator until a prior over code indices is learned.


### **Posterior Collapse** {#posterior-collapse}

Posterior collapse occurs when $q_{\phi}(z\mid x)\approx p(z)$ and the decoder ignores $z$. Then the per-example KL approaches zero, latent means vary little across data, and replacing $z$ has little effect. Powerful autoregressive decoders are especially susceptible because they can model $x$ from previous outputs without using the latent channel.

![An informative posterior changes with the input, whereas a collapsed posterior matches the prior and leaves no active latent channel.](assets/dl15-posterior-collapse.svg){fig-align="center" width="76%" fig-alt="Panels compare informative and collapsed posteriors and list per-dimension KL, active units, and decoder sensitivity diagnostics."}

*Original diagnostic diagram informed by [Lagging Inference Networks and Posterior Collapse](https://arxiv.org/abs/1901.05534).*

KL annealing, free bits, decoder weakening, skip connections from $z$, more expressive posteriors, and extra inference updates can help, but each changes optimization or modeling assumptions. A single nonzero total KL can hide partial collapse, so inspect KL and variance per dimension.

<details>
<summary><strong>PyTorch: diagnose active latent units and decoder sensitivity</strong></summary>

```python
def collapse_diagnostics(model, images):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(images)
        kl_per_dimension = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).mean(0)
        active_units = int((mu.var(0) > 1e-2).sum())
        original = model.decoder(mu)
        shuffled = model.decoder(mu[torch.randperm(len(mu), generator=torch.Generator().manual_seed(1590))])
        decoder_sensitivity = float((original - shuffled).pow(2).mean())
    return kl_per_dimension, active_units, decoder_sensitivity


standard_kl_dims, standard_active, standard_sensitivity = collapse_diagnostics(vae, test_x)
high_kl_dims, high_active, high_sensitivity = collapse_diagnostics(high_beta_vae, test_x)
print({"model": "beta=1", "active units": standard_active,
       "mean KL per dim": round(float(standard_kl_dims.mean()), 3),
       "decoder sensitivity": round(standard_sensitivity, 4)})
print({"model": "beta=4", "active units": high_active,
       "mean KL per dim": round(float(high_kl_dims.mean()), 3),
       "decoder sensitivity": round(high_sensitivity, 4)})

assert standard_active <= vae.latent_dim and high_active <= high_beta_vae.latent_dim
assert standard_sensitivity >= 0 and high_sensitivity >= 0
```

</details>

High $\beta$ is used here to demonstrate capacity pressure, not to claim complete collapse. A proper collapse study tracks diagnostics over training, controls decoder capacity, and tests whether generated outputs change under latent interventions. Low KL can be appropriate when the true task needs little latent information.


### **Normalizing Flows** {#normalizing-flows}

Let $z=f_{\theta}(x)$ be invertible and let $p_Z(z)$ be tractable. The change-of-variables formula gives

$$
\log p_X(x)=\log p_Z(f_{\theta}(x))+log\left|\det\frac{\partial f_{\theta}(x)}{\partial x}\right|.
$$

Unlike a VAE, a flow provides exact latent inference and exact likelihood under its continuous-density assumptions. Invertibility preserves dimensionality and restricts architecture. General Jacobian determinants cost $O(D^3)$, so flow layers use triangular, autoregressive, or structured Jacobians.

RealNVP affine coupling leaves one partition unchanged and uses it to scale and shift the other. Its Jacobian is triangular, making the log determinant a sum of scale outputs; the inverse is analytic.

![A RealNVP affine coupling layer has a triangular Jacobian, tractable log determinant, and exact inverse.](assets/dl15-realnvp.svg){fig-align="center" width="78%" fig-alt="Three panels show affine coupling equations, the change-of-variables density, and the exact inverse."}

*Original derivation diagram based on [Density Estimation using RealNVP](https://arxiv.org/abs/1605.08803).*

<details>
<summary><strong>PyTorch: fit an invertible RealNVP density to the shared two-dimensional codes</strong></summary>

```python
class AffineCoupling(nn.Module):
    def __init__(self, mask):
        super().__init__()
        self.register_buffer("mask", torch.tensor(mask, dtype=torch.float32))
        self.network = nn.Sequential(nn.Linear(2, 48), nn.Tanh(), nn.Linear(48, 48),
                                     nn.Tanh(), nn.Linear(48, 4))

    def forward(self, inputs):
        fixed = inputs * self.mask
        scale, shift = self.network(fixed).chunk(2, dim=1)
        scale = 1.4 * torch.tanh(scale) * (1 - self.mask)
        shift = shift * (1 - self.mask)
        outputs = fixed + (1 - self.mask) * (inputs * torch.exp(scale) + shift)
        return outputs, scale.sum(dim=1)

    def inverse(self, outputs):
        fixed = outputs * self.mask
        scale, shift = self.network(fixed).chunk(2, dim=1)
        scale = 1.4 * torch.tanh(scale) * (1 - self.mask)
        shift = shift * (1 - self.mask)
        return fixed + (1 - self.mask) * (outputs - shift) * torch.exp(-scale)


class RealNVP(nn.Module):
    def __init__(self, layers=6):
        super().__init__()
        self.layers = nn.ModuleList([
            AffineCoupling([1, 0] if index % 2 == 0 else [0, 1]) for index in range(layers)
        ])

    def transform(self, inputs):
        latent, logdet = inputs, torch.zeros(len(inputs))
        for layer in self.layers:
            latent, contribution = layer(latent)
            logdet += contribution
        return latent, logdet

    def log_prob(self, inputs):
        base, logdet = self.transform(inputs)
        base_log_prob = -0.5 * (base.pow(2) + math.log(2 * math.pi)).sum(dim=1)
        return base_log_prob + logdet

    def sample(self, count, seed=1600):
        generator = torch.Generator().manual_seed(seed)
        outputs = torch.randn((count, 2), generator=generator)
        for layer in reversed(self.layers):
            outputs = layer.inverse(outputs)
        return outputs


seed_everything(1600)
flow = RealNVP()
optimizer = torch.optim.AdamW(flow.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1600)
for _ in range(550):
    indices = torch.randint(len(train_z), (256,), generator=generator)
    optimizer.zero_grad()
    loss = -flow.log_prob(train_z[indices]).mean()
    loss.backward()
    optimizer.step()

flow.eval()
with torch.no_grad():
    flow_test_nll = float(-flow.log_prob(test_z).mean())
    flow_samples = flow.sample(300)
    transformed, _ = flow.transform(test_z[:32])
    reconstructed = transformed
    for layer in reversed(flow.layers):
        reconstructed = layer.inverse(reconstructed)
    inversion_error = float((reconstructed - test_z[:32]).abs().max())

assert inversion_error < 1e-4
assert torch.isfinite(flow_samples).all()
print({"test latent NLL": round(flow_test_nll, 3), "max inversion error": inversion_error,
       "sample mean": flow_samples.mean(0).round(decimals=3).tolist(),
       "sample std": flow_samples.std(0).round(decimals=3).tolist()})
```

</details>

The reported likelihood is for two-dimensional autoencoder codes after training-only standardization, not for raw images. Because the deterministic encoder is not invertible, this flow cannot assign image-space likelihood. It is a controlled visualization of flow mechanics and a reminder that likelihood units depend on the modeled space.


### **Energy-Based Models** {#energy-based-models}

An energy-based model defines

$$
p_{\theta}(x)=\frac{\exp[-E_{\theta}(x)]}{Z_{\theta}},\qquad
Z_{\theta}=\int \exp[-E_{\theta}(x)]\,dx.
$$

$E_{\theta}(x)$ can be highly flexible, but the partition function $Z_{\theta}$ couples all possible configurations. Maximum-likelihood gradients contain a positive phase that lowers energy on data and a negative phase that raises energy on model samples. Obtaining the negative phase usually requires MCMC, approximation, or an alternative objective.

The executable example uses noise-contrastive estimation (NCE): a classifier distinguishes data codes from samples of known noise density $q(z)$. At the optimum, its logit estimates a log density ratio, allowing an unnormalized data log density and energy to be recovered up to a constant. This makes the training signal explicit without claiming exact maximum likelihood.

![Energy defines a landscape; its negative gradient is the score, and Langevin dynamics combines score drift with noise.](assets/dl15-energy-score.svg){fig-align="center" width="78%" fig-alt="An energy landscape contains low-energy data basins; a score field points toward higher density and Langevin dynamics adds Gaussian noise."}

*Original synthesis based on the [Energy-Based Learning tutorial](https://yann.lecun.org/exdb/publis/pdf/lecun-06.pdf).*

<details>
<summary><strong>PyTorch: learn a latent energy with NCE and sample it using Langevin dynamics</strong></summary>

```python
class DensityRatio(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64),
                                     nn.SiLU(), nn.Linear(64, 1))

    def forward(self, points):
        return self.network(points).squeeze(1)


noise_scale = 2.2


def noise_log_prob(points):
    return -0.5 * ((points / noise_scale).pow(2) + math.log(2 * math.pi * noise_scale**2)).sum(1)


seed_everything(1610)
ratio_model = DensityRatio()
optimizer = torch.optim.AdamW(ratio_model.parameters(), lr=2e-3, weight_decay=1e-4)
generator = torch.Generator().manual_seed(1610)
for _ in range(650):
    indices = torch.randint(len(train_z), (192,), generator=generator)
    data_batch = train_z[indices]
    noise_batch = noise_scale * torch.randn((192, 2), generator=generator)
    points = torch.cat([data_batch, noise_batch])
    targets = torch.cat([torch.ones(192), torch.zeros(192)])
    optimizer.zero_grad()
    loss = F.binary_cross_entropy_with_logits(ratio_model(points), targets)
    loss.backward()
    optimizer.step()


def unnormalized_log_density(points):
    return ratio_model(points) + noise_log_prob(points)


def langevin_energy_samples(count=300, steps=120, step_size=0.025, seed=1611):
    generator = torch.Generator().manual_seed(seed)
    points = noise_scale * torch.randn((count, 2), generator=generator)
    for _ in range(steps):
        points.requires_grad_(True)
        log_density = unnormalized_log_density(points).sum()
        score = torch.autograd.grad(log_density, points)[0]
        noise = torch.randn(points.shape, generator=generator)
        points = (points + step_size * score + math.sqrt(2 * step_size) * noise).detach().clamp(-6, 6)
    return points


ratio_model.eval()
energy_samples = langevin_energy_samples()
with torch.no_grad():
    data_energy = float((-unnormalized_log_density(test_z)).mean())
    broad_noise = noise_scale * torch.randn((len(test_z), 2), generator=torch.Generator().manual_seed(1612))
    noise_energy = float((-unnormalized_log_density(broad_noise)).mean())

assert data_energy < noise_energy
assert energy_samples.shape == (300, 2)
print({"mean data energy": round(data_energy, 3), "mean broad-noise energy": round(noise_energy, 3),
       "Langevin sample mean": energy_samples.mean(0).round(decimals=3).tolist()})
```

</details>

Short-run Langevin samples can remain biased by initialization, step size, mixing, and multimodal barriers. Energy values are identifiable only up to an additive constant. Diagnostics should include chains from multiple initializations, autocorrelation, effective sample size, and held-out tasks rather than a single attractive scatter plot.


### **Score Matching** {#score-matching}

The score of a continuous density is

$$
s_{\theta}(x)=\nabla_x\log p_{\theta}(x)=-\nabla_x E_{\theta}(x).
$$

The unknown partition function disappears under the gradient. Classical [score matching](https://jmlr.org/papers/v6/hyvarinen05a.html) fits model and data scores without evaluating $Z_{\theta}$. Denoising score matching corrupts $x$ with Gaussian noise and predicts the score of the noisy conditional distribution. For $\tilde{x}=x+\epsilon$, $\epsilon\sim\mathcal{N}(0,\sigma^2I)$, the target is

$$
\nabla_{\tilde{x}}\log q(\tilde{x}\mid x)
=-\frac{\tilde{x}-x}{\sigma^2}=-\frac{\epsilon}{\sigma^2}.
$$

The learned field points noisy points toward higher-density regions of the Gaussian-smoothed data distribution. A single noise level loses fine detail when $\sigma$ is large and is difficult near a thin manifold when $\sigma$ is small. Chapter 16 extends this idea to many noise levels and reverse diffusion.

<details>
<summary><strong>PyTorch: learn a denoising score field on the same latent distribution</strong></summary>

```python
class ScoreNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64),
                                     nn.Tanh(), nn.Linear(64, 2))

    def forward(self, points):
        return self.network(points)


score_sigma = 0.35
seed_everything(1620)
score_model = ScoreNetwork()
optimizer = torch.optim.AdamW(score_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1620)
for _ in range(700):
    indices = torch.randint(len(train_z), (256,), generator=generator)
    clean = train_z[indices]
    epsilon = score_sigma * torch.randn(clean.shape, generator=generator)
    noisy = clean + epsilon
    target_score = -epsilon / score_sigma**2
    optimizer.zero_grad()
    score_loss = F.mse_loss(score_model(noisy), target_score)
    score_loss.backward()
    optimizer.step()

with torch.no_grad():
    validation_epsilon = score_sigma * torch.randn(val_z.shape, generator=torch.Generator().manual_seed(1621))
    validation_score_loss = float(F.mse_loss(
        score_model(val_z + validation_epsilon), -validation_epsilon / score_sigma**2
    ))


def score_langevin(count=300, steps=140, step_size=0.012, seed=1622):
    generator = torch.Generator().manual_seed(seed)
    points = 2.2 * torch.randn((count, 2), generator=generator)
    initial = points.clone()
    for _ in range(steps):
        with torch.no_grad():
            drift = score_model(points)
        noise = torch.randn(points.shape, generator=generator)
        points = (points + step_size * drift + math.sqrt(2 * step_size) * noise).clamp(-6, 6)
    return initial, points


score_initial, score_samples = score_langevin()
initial_distance = torch.cdist(score_initial, train_z).min(1).values.mean()
final_distance = torch.cdist(score_samples, train_z).min(1).values.mean()
assert torch.isfinite(score_samples).all()
print({"validation denoising score MSE": round(validation_score_loss, 3),
       "initial nearest-data distance": round(float(initial_distance), 3),
       "final nearest-data distance": round(float(final_distance), 3)})
```

</details>

The nearest-data distance should be interpreted cautiously: moving close to a training point may indicate realistic structure or memorization. Score accuracy, sampler discretization, and mixing error are separate. A field with low denoising loss at one $\sigma$ is not automatically a calibrated density model across scales.


### **Latent, Flow, and Energy Models Compared** {#latent-flow-energy-models-compared}

These families differ less by “generation quality” than by which computation they make tractable.

| Family | Density access | Inference | Sampling | Main structural cost |
|---|---|---|---|---|
| Deterministic AE | no normalized density | one encoder pass | undefined without a latent prior | reconstruction can leave holes in latent space |
| VAE | ELBO / approximate marginal likelihood | amortized $q_{\phi}(z\mid x)$ | prior sample plus decoder | variational gap and rate-distortion trade-off |
| VQ-VAE | discrete codes; prior learned separately | nearest codebook lookup | requires a code prior | codebook collapse and straight-through bias |
| Normalizing flow | exact continuous likelihood | exact inverse/forward map | exact inverse transform | invertibility, equal dimension, Jacobian design |
| EBM | unnormalized density | optimization or conditional sampling | usually iterative MCMC | partition function and mixing |
| Score model | density gradient, not pointwise density by default | vector field evaluation | iterative stochastic dynamics | noise-scale coverage and numerical solver error |

For the chapter experiment, the autoencoder code is a shared measurement space. The flow has exact likelihood only in that space. The NCE energy and denoising score avoid explicit normalization but rely on approximate sampling. The VAE directly models images through latent uncertainty but optimizes a bound. Comparing their raw loss values would be meaningless because the targets and units differ.

Selection should begin with required operations. Use a VAE when amortized inference and a structured stochastic bottleneck matter; VQ-VAE when discrete reusable units are useful; a flow when exact continuous likelihood and invertibility justify architectural constraints; an EBM when flexible compatibility or conditional inference matters; and a score model when gradients of log density and iterative refinement are central.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

This chapter replaced one ordered factorization with three broader modeling strategies. Latent-variable models compress and explain observations through hidden structure. Variational inference turns an intractable posterior into an optimizable ELBO. Flows preserve exact density through invertibility. Energy and score models trade normalized likelihood for flexible landscapes and gradient-based sampling.

The shared UCI experiment exposed the operational differences. A deterministic bottleneck reconstructed digits but needed an added density before sampling. Denoising and sparsity imposed useful invariances and capacity limits. VAE experiments separated reconstruction from KL rate, verified pathwise gradients, and diagnosed the effect of $\beta$. VQ-VAE made codebook utilization measurable. RealNVP unit-tested invertibility. NCE learned an energy relative to known noise, while denoising score matching learned a local vector field over the same latent distribution.

A practical audit asks: What random variables and observation likelihood are assumed? Is latent inference exact, approximate, or undefined? In which space is likelihood measured? Does sampling require a prior, inverse map, autoregressive prior, or MCMC? Which approximation creates bias: variational family, straight-through gradients, finite flow architecture, contrastive objective, or discretized dynamics? Finally, do reconstruction, held-out density, coverage, memorization, and downstream utility agree?

Chapter 16 moves from these foundations to adversarial and iterative high-fidelity generation. GANs learn through a discriminator, diffusion models learn denoising across noise scales, and flow matching learns continuous transport. The score-matching section here provides the conceptual bridge without duplicating the full diffusion derivation.
